# 실습 5 심화 (선택) · LLM에게 피처 엔지니어링 시키기

05b 비지도 모델의 재현율 0.31을 올리도록 LLM에게 목표 지표와 금지 조건을 주고, 피처 코드를 제안받아 사람이 읽고 확인한 뒤 실행 · 평가합니다.

- 학습셋을 다시 나눠 fit 70% / 검증 30%. 반복 평가는 검증셋으로만, 평가셋은 마지막에 한 번만 봅니다.
- 목표: 검증셋 재현율 0.45 이상, 정밀도 0.30 이상 (경보 5%)
- LLM이 만든 코드를 실행하는 실습입니다. 3일차 '과도한 권한' 위험과 같은 문제라 실행 전 사람 확인이 통제 지점입니다.
- 작은 로컬 모델은 실행되는 코드를 자주 못 만듭니다. 가능하면 `ENV = "mono"`로 합니다.

In [1]:
import os
import numpy as np
import pandas as pd

assert os.path.exists("../data/processed/train.csv"), "05a_data_preprocessing.ipynb를 먼저 실행하세요"
train = pd.read_csv("../data/processed/train.csv", dtype={"dsport": str})
test = pd.read_csv("../data/processed/test.csv", dtype={"dsport": str})
print(f"학습 {len(train)}건 (공격 {train['label'].sum()}) / 평가 {len(test)}건 (공격 {test['label'].sum()})")

학습 3520건 (공격 166) / 평가 1509건 (공격 71)


In [2]:
DESIGNED = ["log_ct_srv_dst", "log_ct_dst_ltm", "log_ct_src_dport_ltm",       # 연결 행위
            "log_sbytes_vs_svc", "log_dbytes_vs_svc", "log_rate_vs_svc",      # 규모 (서비스 평소 대비)
            "byte_ratio_vs_svc", "pkt_ratio_vs_svc"]                          # 방향 (서비스 평소 대비)
CATS = ["DoS", "Reconnaissance", "Exploits"]

In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

full = pd.concat([train.assign(part="train"), test.assign(part="test")], ignore_index=True)
tr_idx = full.index[full["part"] == "train"]
fit_idx, valid_idx = train_test_split(tr_idx, test_size=0.3, stratify=full.loc[tr_idx, "attack_cat"], random_state=7)
test_idx = full.index[full["part"] == "test"]
INPUT = full.drop(columns=["attack_cat", "label", "part"])          # 피처 코드에는 정답을 넘기지 않음

def evaluate(feat, rows):
    """fit 행으로 학습 → rows 행에서 경보 5% 기준 정밀도 · 재현율 · AUC-ROC"""
    X = feat.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)
    sc = StandardScaler().fit(X.loc[fit_idx])
    m = IsolationForest(n_estimators=300, contamination=0.05, random_state=42).fit(sc.transform(X.loc[fit_idx]))
    Xr = sc.transform(X.loc[rows])
    pred, y = (m.predict(Xr) == -1).astype(int), full.loc[rows, "label"]
    return {"정밀도": round(precision_score(y, pred, zero_division=0), 2), "재현율": round(recall_score(y, pred), 2),
            "AUC-ROC": round(roc_auc_score(y, -m.score_samples(Xr)), 3)}

## 기준 성능 (05a 설계 피처 8개, 검증셋)

In [4]:
GOAL = {"재현율": 0.45, "정밀도": 0.30}
base = evaluate(full[DESIGNED], valid_idx)
print("기준:", base, "| 목표:", GOAL)

기준: {'정밀도': 0.3, '재현율': 0.32, 'AUC-ROC': 0.87} | 목표: {'재현율': 0.45, '정밀도': 0.3}


In [5]:
from openai import OpenAI

ENV = "mono"          # "ollama" = 내 PC,  "mono" = MonoRouter(원격)
if ENV == "mono":
    client = OpenAI(base_url="https://monogpt.kr/api/monorouter/v1/openai/v1",
                    api_key="mr_openai_e0di9bdw3iFdUDG1hgAxEfo_tH_DXI2z")
    MODEL = "gpt-5.5"
else:
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    MODEL = "llama3.2:3b"
print(ENV, MODEL)

mono gpt-5.5


In [6]:
def ask(prompt, system=None, temperature=0.2, show=True):
    """질문을 보내고 답을 받아 돌려줌. show=True면 답이 만들어지는 대로 화면에 표시"""
    messages = [{"role": "user", "content": prompt}]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    limit = {"max_tokens": 500} if ENV == "ollama" else {}   # 작은 모델이 끝없이 길게 답하는 것을 막음
    stream = client.chat.completions.create(model=MODEL, messages=messages,
                                            temperature=temperature, stream=True, **limit)
    answer = ""
    for chunk in stream:
        piece = chunk.choices[0].delta.content if chunk.choices else None
        if piece:
            answer += piece
            if show:
                print(piece, end="", flush=True)
    if show:
        print()
    return answer

In [7]:
import json, re

def parse_json(raw):
    """답에서 { ... } 부분만 꺼내 JSON으로 읽음. 없거나 깨졌으면 None"""
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

## 프롬프트: 역할 · 맥락 · 목표 · 제약 · 출력 형식

In [8]:
PROMPT = """[역할] 당신은 보안 데이터 과학자입니다. 비지도 이상탐지(Isolation Forest)의 입력 피처를 설계합니다.
[맥락] UNSW-NB15 네트워크 연결 기록. 한 행 = 연결 1건. 컬럼:
srcip, sport, dstip, dsport(문자열), Stime(유닉스 초), proto, state(FIN 정상 종료, INT 응답 없음), service(-는 식별 안 됨),
dur, spkts, dpkts, sbytes, dbytes, rate, ct_srv_dst, ct_dst_ltm, ct_src_dport_ltm, 05a 설계 피처 8개(log_..., ..._vs_svc)
지금 성능(검증셋, 경보 5%): {score}
지금까지 시도: {history}
[목표] 검증셋 재현율 0.45 이상, 정밀도 0.30 이상
[제약] 정답 컬럼은 없다. IP 값 자체를 피처로 쓰지 않는다(같은 IP끼리 묶어 세는 것은 가능). 미래 시각 정보 금지(과거 시간 창만).
import 금지(pd, np는 이미 있음). 파일 · 네트워크 접근 금지. 함수 이름 make_features(df), 같은 index의 숫자 DataFrame, 피처 20개 이하.
[출력 형식] JSON으로만 답한다.
{{"idea": "한두 문장", "code": "def make_features(df):\\n    ..."}}"""

FORBIDDEN = ["label", "attack_cat", "import", "open(", "exec", "eval", "__", "os.", "sys.", "subprocess",
             "sttl", "dttl", "to_csv", "request", "socket"]

## 한 번 제안받기 → 코드 읽기 → 실행 여부 결정
이 셀을 다시 실행할 때마다 새 제안을 받습니다. 금지어가 있으면 실행하지 않습니다. 코드를 읽고 아래 셀에서 `RUN = True`로 바꿔 실행합니다.

In [9]:
history = []
plan = parse_json(ask(PROMPT.format(score=base, history=history or "없음"), temperature=0.3))
code = plan.get("code", "") if plan else ""
bad = [w for w in FORBIDDEN if w in code]
print("\n아이디어:", plan.get("idea") if plan else "형식 오류")
print("금지어:", bad or "없음")
print("--- 제안 코드 ---\n" + code)

{"idea":"기본 로그/비율 피처에 더해, 같은 src/dst/src-dport의 과거 60~300초 빈도와 service별 과거 log_rate 편차를 넣어 스캔·무응답·서비스 대비 비정상 트래픽을 잡도록 설계합니다. IP 원값은 출력하지 않고 과거 시점 누적/윈도우 카운트로만 사용합니다.","code":"def make_features(df):\n    idx = df.index\n    n = len(df)\n\n    def num(c, default=0.0):\n        if c in df.columns:\n            return pd.to_numeric(df[c], errors='coerce').fillna(default).astype(float)\n        return pd.Series(default, index=idx, dtype=float)\n\n    def cat(c):\n        if c in df.columns:\n            s = df[c]\n            return s.where(s.notna(), '').astype(str)\n        return pd.Series('', index=idx, dtype=object)\n\n    stime = num('Stime', 0.0)\n    order = np.argsort(stime.values, kind='mergesort')\n    times = stime.values[order]\n\n    def past_count(key_like, window):\n        if n == 0:\n            return np.array([], dtype=float)\n        ks = pd.Series(key_like, index=idx)\n        ks = ks.where(ks.notna(), '').astype(str).values[order]\n        out_s = np.zeros(n, dtype=fl

In [11]:
#RUN = False      # 코드를 읽고 실행해도 된다고 판단하면 True
RUN = True
if RUN and code and not bad:
    ns = {"pd": pd, "np": np}
    exec(code, ns)                                   # 사람이 확인한 코드만 실행
    feat = ns["make_features"](INPUT.copy())
    score = evaluate(feat, valid_idx)
    print("검증셋:", score, "| 목표 달성:", score["재현율"] >= GOAL["재현율"] and score["정밀도"] >= GOAL["정밀도"])
    history.append({"idea": plan.get("idea"), "결과": score})

검증셋: {'정밀도': 0.37, '재현율': 0.34, 'AUC-ROC': 0.934} | 목표 달성: False


## 마지막에 한 번만: 평가셋 비교

In [12]:
print("기준(설계 피처 8개):", evaluate(full[DESIGNED], test_idx))
if RUN and code and not bad:
    print("LLM 제안           :", evaluate(feat, test_idx))

기준(설계 피처 8개): {'정밀도': 0.24, '재현율': 0.28, 'AUC-ROC': 0.869}
LLM 제안           : {'정밀도': 0.29, '재현율': 0.34, 'AUC-ROC': 0.927}


### 토론
- 목표를 숫자로 줬을 때와 '성능을 올려 줘'라고만 했을 때 제안이 어떻게 다른지 비교합니다.
- LLM이 정답 · IP 값 · TTL 같은 지름길이나 미래 정보를 쓰려 했나요? 금지어 검사와 사람 확인이 잡아냈나요?
- 검증셋에 맞춰 여러 번 고치면 검증셋 점수만 오릅니다. 평가셋은 마지막에 한 번만 봅니다.
- 강사용 참고 답안: `05s_reference_answer.ipynb` (평가셋 정밀도 0.58 · 재현율 0.61)